# 🧪 PT-W1-D6 概念实验：三层分离验证

> 配套阅读：同名 .md
> 实验目标：模拟 Ontology / Knowledge / RAG 三者协作，验证各自边界

## 第 1 格：三层定义 — 规划图/图书馆/快递

In [ ]:
layers = {
    "Ontology": {
        "metaphor": "城市规划图",
        "answers": "企业里有什么？什么关系？什么规则？",
        "assets": "business-ontology.yaml + ADR-006 + effect-registry",
        "format": "结构化（YAML/声明式）",
    },
    "Knowledge": {
        "metaphor": "图书馆目录",
        "answers": "什么知识需要被检索？",
        "assets": "LangChat AI Knowledge（制度/手册/模板）",
        "format": "非结构化（文档/向量索引）",
    },
    "RAG": {
        "metaphor": "快递系统",
        "answers": "怎么把正确的知识送到 AI 面前？",
        "assets": "向量嵌入 + 混合搜索 + 重排序",
        "format": "检索机制（向量/关键词）",
    },
}

for name, info in layers.items():
    print(f"\n📋 {name}（{info['metaphor']}）")
    print(f"   回答: {info['answers']}")
    print(f"   资产: {info['assets']}")
    print(f"   格式: {info['format']}")

## 第 2 格：三者各自做不到什么

In [ ]:
questions = [
    ("A101 和合同 #2023-085 什么关系？", "Ontology"),
    ("退租操作手册 §3.2 说了什么？", "Knowledge + RAG"),
    ("A101 当前能不能出租？", "Ontology（规则推理）"),
    ("上季度各区域平均客单价？", "SQL（都不是）"),
    ("铺位计租面积怎么算？", "Knowledge + RAG"),
    ("合同终止后影响哪些对象？", "Ontology（effect-registry）"),
]

for q, layer in questions:
    icon = "✅" if layer.startswith("Ontology") else ("📚" if "Knowledge" in layer else "🗄️")
    print(f"  {icon} [{layer}] {q}")

## 第 3 格：完整协作场景 — A101 为什么不能出租

In [ ]:
def answer_why_not_leasable(ontology_rules, knowledge_snippets):
    answer_parts = []
    answer_parts.append("【Ontology 推理 — 结构语义】")
    for rule in ontology_rules:
        answer_parts.append(f"  • {rule}")
    answer_parts.append("\n【Knowledge + RAG — 文档检索】")
    for snippet in knowledge_snippets:
        answer_parts.append(f"  • {snippet}")
    return "\n".join(answer_parts)

ontology_rules = [
    "A101 关联 Lease #2023-085，状态 = Terminating",
    "Rule: 存在未完成退租流程的 Space 不可出租",
    "Effect: Lease.terminated → occupancy-effect → Space: pending_release",
]
knowledge_snippets = [
    "《退租管理操作手册》§3.2: 流程 = 合同终止 → 场地验收 → 费用结算 → 铺位释放",
    "《2026 招商政策》: 退租验收未完成的铺位不列入可招商清单",
]

print(answer_why_not_leasable(ontology_rules, knowledge_snippets))
print("\n⚠️ Ontology 提供骨架，Knowledge + RAG 提供血肉，Agent 组合成完整回答")

## 第 4 格：Ontology 如何指导 Knowledge 组织

In [ ]:
print("【无 Ontology 指导】Knowledge Base：")
print("  一大堆文档 → 统一切片 → 向量入库 → 暴力检索")
print("  问题：检索到什么全靠运气")
print()
print("【有 Ontology 指导】Knowledge Base：")
print("  1. Ontology 定义 14 个业务域 → 文档按域归类")
print("  2. Ontology 定义实体生命周期 → 文档标注适用状态")
print("  3. 检索时按域+状态过滤 = 更精准")
print()
doc_metadata = [
    {"doc": "退租操作手册", "domain": "租赁管理", "applies_to": "Lease.status ∈ {Terminating}"},
    {"doc": "招商政策", "domain": "招商管理", "applies_to": "Space.available"},
]
print("文档元数据标注示例：")
for meta in doc_metadata:
    print(f"  {meta['doc']}: domain={meta['domain']}, applies_to={meta['applies_to']}")

## 第 5 格：三层架构可视化

In [ ]:
from matplotlib import font_manager, pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 7))
boxes = [
    (0.5, 5.5, 3, 1.2, "#2ecc71", "Ontology 层\n企业结构语义\n(有什么/什么关系/什么规则)"),
    (0.5, 3.5, 3, 1.2, "#3498db", "Knowledge 层\n企业知识内容\n(制度/手册/模板)"),
    (0.5, 1.5, 3, 1.2, "#f39c12", "RAG 层\n检索增强机制\n(向量检索+重排序)"),
    (0.5, -0.3, 3, 1.2, "#e74c3c", "Agent 执行层\nSkill Release\n(业务动作)"),
]
for x, y, w, h, color, label in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
            facecolor=color, alpha=0.7)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha="center", va="center", fontsize=9, fontweight="bold")

arrows = [(0.5+1.5, 5.5, 0.5+1.5, 4.7, "指导"),
          (0.5+1.5, 3.5, 0.5+1.5, 2.7, "检索"),
          (0.5+1.5, 1.5, 0.5+1.5, 0.9, "增强")]
for x1, y1, x2, y2, label in arrows:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=2))
    ax.text(x1+0.15, (y1+y2)/2, label, fontsize=8, color="#7f8c8d")

ax.set_xlim(0, 4.5); ax.set_ylim(-0.8, 7.2)
ax.axis("off")
ax.set_title("Ontology / Knowledge / RAG / Agent 四层架构", fontsize=14)
plt.tight_layout()
plt.savefig("/tmp/w1d6_layers.png", dpi=120)
plt.show()
print("三者各自做不到对方的事 → 不能互相替代，只能协作")